In [0]:
%run "/Users/patelrahul2614@gmail.com/databrick_demo/Digital_Banking_LakeHouse_Capstone/includes"

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# Read the account table from the bronze layer
account_df = spark.table(f"{catalog}.bronze.accounts")

# Drop meta columns: file name, path, and ingestion date
account_df = account_df.drop("file_name", "file_path", "ingestion_date")

# Drop fully duplicate rows across all columns, then duplicates by account_id (keep first occurrence)
account_df = account_df.dropDuplicates().dropDuplicates(["account_id"])

critical_columns = ["account_id", "account_type", "account_status", "opening_date"]

null_counts = account_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in critical_columns
])


# Additional validation
# 1. Drop rows with nulls in any critical column
account_df = account_df.na.drop(subset=critical_columns)

# 2. Ensure interest_rate is non-negative
account_df = account_df.filter((col("interest_rate") >= 0) | col("interest_rate").isNull())

# 3. Ensure opening_date is a valid date (cast and filter invalid)
account_df = account_df.withColumn("opening_date", to_date(col("opening_date")))

# 4. Ensure account_status is one of the allowed values
valid_statuses = ["Active", "Dormant", "Closed"]
account_df = account_df.filter(col("account_status").isin(valid_statuses) | col("account_status").isNull())

#5. taking customer and branch to validdate accounts
customer_df = spark.table(f"{catalog}.silver.silver_customers")
branch_df = spark.table(f"{catalog}.silver.silver_branches")

# 6. Validate customer_id and branch_id against silver tables
valid_customer_ids = customer_df.select("customer_id").distinct()
valid_branch_ids = branch_df.select("branch_id").distinct()

# Flag each account row: does customer_id and branch_id exist in silver?
account_df = (
    account_df
    .join(valid_customer_ids.withColumn("_customer_valid", lit(True)), on="customer_id", how="left")
    .join(valid_branch_ids.withColumn("_branch_valid", lit(True)), on="branch_id", how="left")
    .withColumn("_customer_valid", coalesce(col("_customer_valid"), lit(False)))
    .withColumn("_branch_valid", coalesce(col("_branch_valid"), lit(False)))
)

# Dropped accounts: rows where customer_id OR branch_id is NOT found in silver
dropped_accounts_df = (
    account_df
    .filter(~col("_customer_valid") | ~col("_branch_valid"))
    .withColumn("drop_reason",
        when(~col("_customer_valid") & ~col("_branch_valid"), lit("invalid_customer_and_branch"))
        .when(~col("_customer_valid"), lit("invalid_customer_id"))
        .otherwise(lit("invalid_branch_id")))
    .drop("_customer_valid", "_branch_valid")
)

# Valid accounts: rows where BOTH customer_id AND branch_id exist in silver
account_df = (
    account_df
    .filter(col("_customer_valid") & col("_branch_valid"))
    .drop("_customer_valid", "_branch_valid")
)

# 7. Save dropped accounts to catalog.silver.dropped_accounts
dropped_accounts_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.dropped_accounts")

# 8. Save cleaned valid accounts to catalog.silver.silver_accounts
account_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.silver_accounts")
